# I will use the following notebook to demonstrate different steps in preprocessing

## These steps will include:

### 1) Slice timing correction
### 2) Motion correction
### 3) Coregistration
### 4) Spatial and temporal filtering

In [1]:
import os
from glob import glob

# Import new things that we'll need
import pandas as pd
import numpy as np
import nipype.interfaces.afni as afni
import nipype.interfaces.fsl as fsl
import nipype.interfaces.freesurfer as fs
from nipype.interfaces.utility import Function
import seaborn as sns
import nibabel as nb
import json
import nipype.interfaces.io as nio
import nipype.pipeline.engine as pe 
import matplotlib.pyplot as plt

%matplotlib inline

### I next want to get a list of all of my functional files

In [2]:
sid = ['021']
base_dir = f'/home/mali092/Mattfeld_PSB6351'
work_dir = f'/scratch/classroom/psb6351/mali092/Mattfeld_PSB6351'
func_dir = os.path.join(base_dir, f'dset/sub-{sid[0]}/ses-S1/func')
fmap_dir = os.path.join(base_dir, f'dset/sub-{sid[0]}/ses-S1/fmap')
fs_dir = os.path.join(base_dir, 'derivatives', 'freesurfer')

# Get a list of my study task json and nifti converted files
func_json = sorted(glob(func_dir + '/*.json'))
func_files = sorted(glob(func_dir + '/*.nii.gz'))
fmap_files = sorted(glob(fmap_dir + '/sub-*_dir-*_epi.nii.gz'))

In [3]:
fmap_files[0]

'/home/mali092/Mattfeld_PSB6351/dset/sub-021/ses-S1/fmap/sub-021_acq-dMRI_dir-AP_epi.nii.gz'

### Next I want to build and run function to perform slice timing correction. I'm going to have to extract some important information from the .json files like the multiband slicetiming.

In [21]:
# Here I am building a function that eliminates the
# mapnode directory structure and assists in saving
# all of the outputs into a single directory
def get_subs(func_files):
    '''Produces Name Substitutions for Each Contrast'''
    subs = []
    for curr_run in range(len(func_files)):
        subs.append(('_subject_id_%s' %curr_run, ''))
    return subs

# Here I am building a function that takes in a
# text file that includes the number of outliers
# at each volume and then finds which volume (e.g., index)
# has the minimum number of outliers (e.g., min) 
# searching over the first 201 volumes
# If the index function returns a list because there were
# multiple volumes with the same outlier count, pick the first one
def best_vol(outlier_count):
    best_vol_num = outlier_count.index(min(outlier_count[:200]))
    if isinstance(best_vol_num, list):
        best_vol_num = best_vol_num[0]
    return best_vol_num

# Here I am creating a list of lists containing the slice timing for each study run
def slc_time(json_file_list):
    import json

    slice_timing_list = []

    for curr_json in json_file_list:
        with open(curr_json, 'r') as curr_json_data:
            curr_func_metadata = json.load(curr_json_data)

        slice_timing_list.append(curr_func_metadata['SliceTiming'])

    return slice_timing_list
# Here I am establishing a nipype work flow that I will eventually execute
psb6351_wf = pe.Workflow(name='psb6351_wf')
psb6351_wf.base_dir = work_dir + f'/psb6351workdir/sub-{sid[0]}'
psb6351_wf.config['execution']['use_relative_paths'] = True

# Create a Function node to substitute names of files created during pipeline
getsubs = pe.Node(Function(input_names=['func_files'],
                           output_names=['subs'],
                           function=get_subs),
                  name='getsubs')
getsubs.inputs.func_files = func_files

# Here I am inputing just the first run functional data
# I want to use afni's 3dToutcount to find the number of 
# outliers at each volume.  I will use this information to
# later select the earliest volume with the least number of outliers
# to serve as the base for the motion correction
id_outliers = pe.Node(afni.OutlierCount(),
                      name = 'id_outliers')
id_outliers.inputs.automask = True
id_outliers.inputs.legendre = True
id_outliers.inputs.polort = 4
id_outliers.inputs.out_file = 'outlier_file'
id_outliers.inputs.in_file = func_files[0]
# USE AFNI'S OUTLIER COUNT TO GET NUMBER OF OUTLIERS PER RUN

#ATM ONLY: Add an unwarping mapnode here using the field maps

 ## Using AP and PA reverse phase-encoded EPI images to estimate
 ## susceptibility distortion with AFNI 3dQwarp

#EXTRA CREDIT FIGURE OUT HOW TO ADD A NODE FOR AFNIS QWARP
#TO UTILIZE THE DISTORTION CORRECTION FIELD MAPS

fmap_ap = glob(fmap_dir + f'/sub-{sid[0]}_dir-AP_epi.nii.gz')[0]
fmap_pa = glob(fmap_dir + f'/sub-{sid[0]}_dir-PA_epi.nii.gz')[0]

print("AP fieldmap:", fmap_ap)
print("PA fieldmap:", fmap_pa)

qwarp = pe.Node(afni.Qwarp(),
                name='qwarp')

qwarp.inputs.base_file = fmap_ap
qwarp.inputs.in_file = fmap_pa
qwarp.inputs.plusminus = True
qwarp.inputs.outputtype = 'NIFTI_GZ'

#NOW CREATE ANOTHER NODE TO APPLY THE WARPS

 ## Applying the distortion correction warp to each functional run

applywarp = pe.MapNode(afni.NwarpApply(),
                       iterfield=['in_file'],
                       name='applywarp')

applywarp.inputs.in_file = func_files
 
psb6351_wf.connect(qwarp, 'source_warp',
                   applywarp, 'warp')

# Create a Function node to identify the best volume based
# on the number of outliers at each volume. I'm searching
# for the index in the first 201 volumes that has the 
# minimum number of outliers and will use the min() function
# I will use the index function to get the best vol. 

getbestvol = pe.Node(Function(input_names=['outlier_count'],
                              output_names=['best_vol_num'],
                              function=best_vol),
                     name='getbestvol')

psb6351_wf.connect(id_outliers, 'out_file',
                   getbestvol, 'outlier_count')
# Extract the earliest volume with the
# the fewest outliers of the first run as the reference 

#USE FSL EXTRACT ROI TO SELECT THE VOLUME FOR MOTION CORRECTION

extractref = pe.Node(fsl.ExtractROI(t_size=1),
                     name='extractref')

extractref.inputs.in_file = func_files[0]

psb6351_wf.connect(getbestvol, 'best_vol_num',
                   extractref, 't_min')

# Below is the command that runs AFNI's 3dvolreg command.
# this is the node that performs the motion correction
# I'm iterating over the functional files which I am passing
# functional data from the slice timing correction node before
# I'm using the earliest volume with the least number of outliers
# during the first run as the base file to register to.

#USE AFNIs 3dvolreg to perform the motion correction.
volreg = pe.MapNode(afni.Volreg(),
                    iterfield=['in_file'],
                    name='volreg')

volreg.inputs.outputtype = 'NIFTI_GZ'
volreg.inputs.zpad = 4
volreg.inputs.in_file = func_files

psb6351_wf.connect(extractref, 'roi_file',
                   volreg, 'basefile')
# Below is the command that runs AFNI's 3dTshift command
# this is the node that performs the slice timing correction
# I input the study func files as a list and the slice timing 
# as a list of lists. I'm using a MapNode to iterate over the two.
# this should allow me to parallelize this on the HPC

#USE AFNIs 3dTSHIFT TO DO THE SLICE TIMING CORRECTION

getslctime = pe.Node(Function(input_names=['json_file_list'],
                              output_names=['slctime_list'],
                              function=slc_time),
                     name='getslctime')

getslctime.inputs.json_file_list = func_json

tshifter = pe.MapNode(afni.TShift(),
                      iterfield=['in_file', 'slice_timing'],
                      name='tshifter')

tshifter.inputs.tr = '1.76'
tshifter.inputs.outputtype = 'NIFTI_GZ'

psb6351_wf.connect(volreg, 'out_file',
                   tshifter, 'in_file')

psb6351_wf.connect(getslctime, 'slctime_list',
                   tshifter, 'slice_timing')

# Calculate the transformation matrix from EPI space to FreeSurfer space
# using the BBRegister command

# USE FREESURFERs BBREGISTER TO DO THE COREGISTRATION


bbregister = pe.Node(fs.BBRegister(),
                     name='bbregister')

# FreeSurfer recon-all directory
bbregister.inputs.subjects_dir = fs_dir
bbregister.inputs.subject_id = f'sub-{sid[0]}'
bbregister.inputs.contrast_type = 'bold'
bbregister.inputs.init = 'fsl'
bbregister.inputs.registered_file = True

# Use the previously selected reference EPI volume
psb6351_wf.connect(extractref, 'roi_file',
                   bbregister, 'source_file')

# ACCESS FREESURFER RECON-ALL OUTPUTS

fssource = pe.Node(nio.FreeSurferSource(),
                   name='fssource')

fssource.inputs.subjects_dir = fs_dir
fssource.inputs.subject_id = f'sub-{sid[0]}'

# CREATE A BINARY BRAIN MASK FROM APARC+ASEG AND DILATE BY 1 VOXEL

binarize = pe.Node(fs.Binarize(),
                   name='binarize')
binarize.inputs.min = 0.5
binarize.inputs.dilate = 1
psb6351_wf.connect(fssource, 'aparc_aseg',
                   binarize, 'in_file')

## TRANSFORM THE FREESURFER BRAIN MASK INTO FUNCTIONAL EPI SPACE

transform_mask = pe.Node(fs.ApplyVolTransform(),
                         name='transform_mask')

# Apply the BBRegister transformation in the reverse direction:
# FreeSurfer anatomical space to EPI space
transform_mask.inputs.inverse = True
transform_mask.inputs.interp = 'nearest'
psb6351_wf.connect(binarize, 'binary_file',
                   transform_mask, 'target_file')
psb6351_wf.connect(extractref, 'roi_file',
                   transform_mask, 'source_file')

# Use the registration calculated by BBRegister
psb6351_wf.connect(bbregister, 'out_reg_file',
                   transform_mask, 'reg_file')

# APPLYING THE EPI-SPACE BRAIN MASK TO THE FUNCTIONAL IMAGES

maskfunc = pe.MapNode(fsl.ApplyMask(),
                      iterfield=['in_file'],
                      name='maskfunc')

# Functional images after motion correction and slice-timing correction
psb6351_wf.connect(tshifter, 'out_file',
                   maskfunc, 'in_file')

# Use the brain mask that has been transformed into EPI space
psb6351_wf.connect(transform_mask, 'transformed_file',
                   maskfunc, 'mask_file')

# Below is the node that collects all the data and saves
# the outputs that I am interested in. Here in this node
# I use the substitutions input combined with the earlier
# function to get rid of nesting
#SAVE EVERYTHING WITH DATASINK

datasink = pe.Node(nio.DataSink(), name="datasink")


# The following two lines set a work directory outside of my 
# local git repo and runs the workflow
psb6351_wf.run(plugin='SLURM',
               plugin_args={'sbatch_args': ('--partition default-part --qos classroom --account acc_psb6351'),
                            'overwrite':True})

AP fieldmap: /home/mali092/Mattfeld_PSB6351/dset/sub-021/ses-S1/fmap/sub-021_dir-AP_epi.nii.gz
PA fieldmap: /home/mali092/Mattfeld_PSB6351/dset/sub-021/ses-S1/fmap/sub-021_dir-PA_epi.nii.gz
261019-10:31:49,197 nipype.workflow INFO:
	 Workflow psb6351_wf settings: ['check', 'execution', 'logging', 'monitoring']
261019-10:31:49,211 nipype.workflow INFO:
	 Running in parallel.
261019-10:31:49,214 nipype.workflow INFO:
	 Pending[0] Submitting[4] jobs Slots[inf]
261019-10:31:49,214 nipype.workflow INFO:
	 Submitting: psb6351_wf.qwarp ID: 0
261019-10:31:49,217 nipype.workflow INFO:
	 [Job 0] Cached (psb6351_wf.qwarp).
261019-10:31:49,218 nipype.workflow INFO:
	 Finished submitting: psb6351_wf.qwarp ID: 0
261019-10:31:49,218 nipype.workflow INFO:
	 Submitting: psb6351_wf.id_outliers ID: 1
261019-10:31:49,220 nipype.workflow INFO:
	 [Job 1] Cached (psb6351_wf.id_outliers).
261019-10:31:49,220 nipype.workflow INFO:
	 Finished submitting: psb6351_wf.id_outliers ID: 1
261019-10:31:49,221 nipype.w

TypeError: can only concatenate str (not "list") to str

In [22]:

# VISUAL QC OF EPI-TO-ANATOMICAL COREGISTRATION

# Load the anatomical image
anat_file = f'/home/mali092/Mattfeld_PSB6351/derivatives/freesurfer/sub-{sid[0]}/mri/brain.mgz'

# Get the BBRegister registered EPI output
registered_epi = bbregister.result.outputs.registered_file

anat_img = nib.load(anat_file)
epi_img = nib.load(registered_epi)

anat_data = anat_img.get_fdata()
epi_data = epi_img.get_fdata()

# Choose approximately the middle sagittal, coronal, and axial slices
sag = anat_data.shape[0] // 2
cor = anat_data.shape[1] // 2
axi = anat_data.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sagittal
axes[0].imshow(np.rot90(anat_data[sag, :, :]), cmap='gray')
axes[0].imshow(np.rot90(epi_data[sag, :, :]), cmap='hot', alpha=0.4)
axes[0].set_title('Sagittal')
axes[0].axis('off')

# Coronal
axes[1].imshow(np.rot90(anat_data[:, cor, :]), cmap='gray')
axes[1].imshow(np.rot90(epi_data[:, cor, :]), cmap='hot', alpha=0.4)
axes[1].set_title('Coronal')
axes[1].axis('off')

# Axial
axes[2].imshow(np.rot90(anat_data[:, :, axi]), cmap='gray')
axes[2].imshow(np.rot90(epi_data[:, :, axi]), cmap='hot', alpha=0.4)
axes[2].set_title('Axial')
axes[2].axis('off')

plt.suptitle('BBRegister Coregistration QC')
plt.tight_layout()
plt.show()

FileNotFoundError: /tmp/tmpl_j_v67t/psb6351_wf/bbregister/result_bbregister.pklz

### Now I will load and plot the motion files

In [23]:
motion_dir = os.path.join(base_dir, f'derivatives/sub-{sid[0]}/motion')

study_motion_files = sorted(glob(motion_dir + '/*.1D'))

for curr_mot_file in study_motion_files:

    motion_data = np.loadtxt(curr_mot_file)

    plt.figure(figsize=(12, 5))
    plt.plot(motion_data)

    plt.xlabel('Volume')
    plt.ylabel('Motion parameter')
    plt.title(os.path.basename(curr_mot_file))
    plt.legend(['Roll', 'Pitch', 'Yaw', 'dS', 'dL', 'dP'])

    plt.show()
    


In [19]:
# PLOT THE MOTION PARAMETERS FOR EACH FUNCTIONAL RUN USING SEABORN

import seaborn as sns

motion_dir = os.path.join(base_dir, f'derivatives/sub-{sid[0]}/motion')

study_motion_files = sorted(glob(motion_dir + '/*.1D'))

motion_labels = ['Roll', 'Pitch', 'Yaw', 'dS', 'dL', 'dP']

for curr_mot_file in study_motion_files:

    # Load motion parameters
    motion_data = np.loadtxt(curr_mot_file)

    # Create a new figure for each functional run
    plt.figure(figsize=(12, 5))

    # Plot each of the six motion parameters
    for i in range(motion_data.shape[1]):
        sns.lineplot(x=np.arange(motion_data.shape[0]),
                     y=motion_data[:, i],
                     label=motion_labels[i])

    plt.xlabel('Volume')
    plt.ylabel('Motion parameter')
    plt.title(os.path.basename(curr_mot_file))
    plt.legend()
    plt.show()